# End-to-End ETL Pipeline

This notebook demonstrates a complete **ETL (Extract, Transform, Load)** pipeline using:
- Pandas
- NumPy
- Scikit-learn

The pipeline:
1. Generates a synthetic dataset
2. Preprocesses and transforms the data
3. Saves the processed dataset to disk

In [ ]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

## Data Generation (Extract Stage)

In [ ]:
def generate_data(n_rows: int = 1000) -> pd.DataFrame:
    """
    Generates a synthetic dataset for ETL testing.
    """
    np.random.seed(42)

    departments = ["Sales", "Engineering", "HR", "Management", "Marketing"]
    education_levels = ["Bachelors", "Masters", "PhD"]

    data = {
        "age": np.random.randint(22, 60, size=n_rows),
        "salary": np.random.randint(40000, 120000, size=n_rows),
        "department": np.random.choice(departments, size=n_rows),
        "education": np.random.choice(education_levels, size=n_rows),
        "experience": np.random.randint(0, 35, size=n_rows),
        "target": np.random.choice([0, 1], size=n_rows)
    }

    df = pd.DataFrame(data)

    # Introduce missing values (10%)
    for col in ["salary", "education", "experience"]:
        df.loc[df.sample(frac=0.1).index, col] = np.nan

    return df

# Generate dataset
df = generate_data(1000)
df.head()

## Build Preprocessing Pipeline (Transform Stage)

In [ ]:
def build_preprocessing_pipeline(df: pd.DataFrame, target_column: str):
    """
    Builds preprocessing pipelines for numeric and categorical features.
    """
    numeric_features = df.select_dtypes(include=["int64", "float64"]).columns.drop(target_column)
    categorical_features = df.select_dtypes(include=["object"]).columns

    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ])

    return preprocessor

## Apply Transformation and Load Data

In [ ]:
# Separate features and target
X = df.drop(columns=["target"])

# Build and apply pipeline
preprocessor = build_preprocessing_pipeline(df, "target")
X_transformed = preprocessor.fit_transform(X)

# Convert to DataFrame
feature_names = preprocessor.get_feature_names_out()
processed_df = pd.DataFrame(X_transformed, columns=feature_names)

# Save processed data
processed_df.to_csv("processed_data.csv", index=False)

processed_df.head()

## ✅ ETL Pipeline Completed

The processed dataset has been saved as `processed_data.csv`.